# 03 - Large-scale circulation indices

**Reviewer point addressed:** *"correlate snow trends with ... circulation indices"*.

Builds a monthly and seasonal table of the teleconnection patterns that plausibly modulate
winter precipitation and temperature over the Zagros / Kurdistan Region of Iraq.

| index | why it matters for the KRI | source |
|---|---|---|
| **NAO** | controls the Atlantic storm track that feeds Mediterranean cyclones | NOAA CPC |
| **AO** | hemispheric polar-vortex strength; cold-air outbreaks | NOAA CPC |
| **EA** (East Atlantic) | modulates the southern flank of the storm track | NOAA CPC |
| **EA/WR** (East Atlantic / West Russia) | **the strongest documented control on Middle East winter precipitation** | NOAA CPC |
| **SCAND** (Scandinavia / blocking) | blocking highs steer cyclones towards the Levant | NOAA CPC |
| **POL** (Polar / Eurasia) | Siberian High strength, cold advection | NOAA CPC |
| **Nino 3.4** | ENSO teleconnection to SW Asia winter rainfall | NOAA PSL |
| **DMI** (Indian Ocean Dipole) | moisture supply to the Arabian Sea / Gulf | NOAA PSL |
| **AMO** | multidecadal Mediterranean drying background | NOAA PSL |
| **NCP** (North Sea - Caspian Pattern) | *optional, computed below* - dipole directly over the study latitude | NCEP/NCAR |
| **MOI** (Mediterranean Oscillation) | *optional, computed below* - Algiers-Cairo SLP dipole | NCEP/NCAR |

Raw text files are cached in `data/indices/`. If you are offline the notebook uses the cache;
if you are online it refreshes it.

In [1]:
import os, re, urllib.request
import numpy as np
import pandas as pd

ROOT   = os.path.abspath(os.path.join(os.getcwd(), ".."))
OUTDIR = os.path.join(ROOT, "data", "derived")
IDXDIR = os.path.join(ROOT, "data", "indices")
os.makedirs(IDXDIR, exist_ok=True)

def fetch(url, fname, timeout=40, refresh=True):
    """Download to data/indices/<fname>; fall back to the cached copy if offline."""
    path = os.path.join(IDXDIR, fname)
    if refresh:
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r:
                data = r.read()
            with open(path, "wb") as f:
                f.write(data)
            print(f"downloaded  {fname}")
            return path
        except Exception as e:
            print(f"download failed ({type(e).__name__}) - using cache for {fname}")
    if not os.path.exists(path):
        raise FileNotFoundError(f"no cached copy of {fname}; connect to the internet once")
    print(f"cached      {fname}")
    return path

## 1. NOAA CPC Northern Hemisphere teleconnection table

One fixed-width file gives NAO, EA, WP, EP/NP, PNA, EA/WR, SCA, TNH, POL and PT.
Missing values are coded `-99.90` (pattern not a leading mode that month).

In [2]:
TELE_URL = "https://ftp.cpc.ncep.noaa.gov/wd52dg/data/indices/tele_index.nh"
p = fetch(TELE_URL, "tele_index_nh.txt")

# The file is nominally fixed-width, but the leading-space convention changes part-way
# through the archive and negative values run into the -99.90 fill code
# (e.g. "0.16-99.90"). A regex parse is safer than read_fwf here.
NAMES = ["NAO", "EA", "WP", "EPNP", "PNA", "EAWR", "SCAND", "TNH", "POL", "PT"]
rows = []
with open(p) as f:
    for ln in f:
        m = re.match(r"\s*(\d{4})\s+(\d{1,2})\s+(.*)$", ln)
        if not m:
            continue
        vals = re.findall(r"-?\d+\.\d+", m.group(3))
        if len(vals) < 11:          # 10 patterns + explained variance
            continue
        rows.append([int(m.group(1)), int(m.group(2))] + [float(v) for v in vals[:10]])

tele = pd.DataFrame(rows, columns=["year", "month"] + NAMES)
tele[NAMES] = tele[NAMES].mask(tele[NAMES] <= -99)     # pattern not a leading mode
print(tele.shape, tele["year"].min(), "-", tele["year"].max())
print("missing per pattern:")
print(tele[NAMES].isna().mean().mul(100).round(1).to_string())
tele.tail(3)

downloaded  tele_index_nh.txt
(919, 12) 1950 - 2026
missing per pattern:
NAO       0.0
EA        0.0
WP        0.0
EPNP      8.3
PNA       0.0
EAWR      0.0
SCAND     0.0
TNH      75.0
POL       0.0
PT       83.5


,year,month,NAO,EA,WP,EPNP,PNA,EAWR,SCAND,TNH,POL,PT
916,2026,5,-0.64,0.70,0.97,-0.65,-1.03,-0.43,-0.43,NaN,0.39,NaN
917,2026,6,0.41,0.85,-2.05,1.57,-0.42,-1.51,-2.40,NaN,-2.49,NaN
918,2026,7,-0.28,0.99,-0.69,-2.47,0.16,-1.78,-1.14,NaN,1.33,NaN


## 2. AO (CPC) and the PSL year x 12 files (Nino 3.4, DMI, AMO)

In [3]:
# --- Arctic Oscillation: three columns (year, month, value)
p = fetch("https://www.cpc.ncep.noaa.gov/products/precip/CWlink/daily_ao_index/"
          "monthly.ao.index.b50.current.ascii", "ao_cpc.txt")
ao = pd.read_csv(p, sep=r"\s+", header=None, names=["year", "month", "AO"])
ao[["year", "month"]] = ao[["year", "month"]].astype(int)

def read_psl_matrix(path, name):
    """PSL layout: line 1 = 'startyear endyear', then 'year v1 ... v12',
    then a trailer that starts with the missing-value code."""
    rows, missing = [], None
    with open(path) as f:
        lines = [ln.rstrip("\n") for ln in f]
    for i, ln in enumerate(lines[1:], start=1):
        parts = ln.split()
        if len(parts) == 13:
            try:
                rows.append([float(x) for x in parts])
            except ValueError:
                break
        elif len(parts) == 1 and rows and missing is None:
            try:
                missing = float(parts[0])
            except ValueError:
                pass
            break
        elif rows:
            break
    d = pd.DataFrame(rows, columns=["year"] + list(range(1, 13)))
    d = d.melt(id_vars="year", var_name="month", value_name=name)
    if missing is not None:
        d.loc[np.isclose(d[name], missing), name] = np.nan
    d.loc[d[name] <= -99, name] = np.nan
    d[["year", "month"]] = d[["year", "month"]].astype(int)
    return d

psl = {
    "NINO34": ("https://psl.noaa.gov/data/correlation/nina34.anom.data", "nino34_psl.txt"),
    "DMI":    ("https://psl.noaa.gov/gcos_wgsp/Timeseries/Data/dmi.had.long.data", "dmi_psl.txt"),
    "AMO":    ("https://psl.noaa.gov/data/correlation/amon.us.data", "amo_psl.txt"),
}
psl_frames = []
for name, (url, fname) in psl.items():
    psl_frames.append(read_psl_matrix(fetch(url, fname), name))

monthly_idx = tele.merge(ao, on=["year", "month"], how="outer")
for d in psl_frames:
    monthly_idx = monthly_idx.merge(d, on=["year", "month"], how="outer")
monthly_idx = monthly_idx.sort_values(["year", "month"]).reset_index(drop=True)
print(monthly_idx.shape)
monthly_idx[(monthly_idx.year == 2020)].round(2)

downloaded  ao_cpc.txt
downloaded  nino34_psl.txt
downloaded  dmi_psl.txt
downloaded  amo_psl.txt
(1884, 16)


,year,month,NAO,EA,WP,EPNP,PNA,EAWR,SCAND,TNH,POL,PT,AO,NINO34,DMI,AMO
1800,2020,1,1.05,1.74,0.69,-0.60,-0.95,0.66,-0.55,-0.87,0.16,NaN,2.42,0.84,0.17,0.07
1801,2020,2,0.98,1.38,1.46,-1.79,-0.07,-0.06,-2.69,1.69,-0.39,NaN,3.42,0.67,0.05,0.33
1802,2020,3,0.66,-0.11,1.29,0.44,-2.41,0.57,-0.88,NaN,1.75,NaN,2.64,0.65,0.02,0.35
1803,2020,4,-1.26,0.58,-1.34,1.54,-1.38,1.82,-1.52,NaN,0.45,NaN,0.93,0.66,-0.01,0.35
1804,2020,5,-0.33,0.06,0.12,0.04,0.27,-0.55,-2.37,NaN,-1.11,NaN,-0.03,0.05,0.30,0.24
1805,2020,6,0.16,-0.08,-1.25,-0.69,0.86,-2.01,0.58,NaN,-0.24,NaN,-0.12,-0.05,0.46,0.25
1806,2020,7,-1.19,0.46,-0.54,-1.97,1.20,-0.68,-2.29,NaN,-0.08,NaN,-0.41,0.08,0.32,0.34
1807,2020,8,0.03,1.57,-0.21,-2.39,1.80,0.64,-1.56,NaN,-0.51,0.54,-0.38,-0.29,-0.18,0.42
1808,2020,9,1.11,1.95,-2.44,0.06,0.59,-0.94,-0.47,NaN,0.32,1.53,0.63,-0.66,-0.19,0.30
1809,2020,10,-0.20,-0.23,-1.18,0.57,-1.08,-1.82,1.45,NaN,-1.01,NaN,-0.07,-0.90,0.07,0.29


## 3. Optional: compute NCP and MOI from NCEP/NCAR reanalysis

Neither index is distributed as a ready-made file, but both are simple two-point pressure
dipoles and are worth having because they sit *directly over* the study region.

- **NCP** (Kutiel & Benaroch, 2002): standardised 500 hPa geopotential height at
  (0 E, 55 N) minus (60 E, 55 N).
- **MOI** (Palutikof, 2003): standardised sea-level pressure at Algiers (3 E, 36.5 N)
  minus Cairo (31.5 E, 30 N).

Requires `xarray` + `netCDF4` and an internet connection (OPeNDAP). Skip this cell if either
is unavailable - everything downstream tolerates the two columns being absent.

```
pip install xarray netCDF4
```

In [4]:
COMPUTE_NCP_MOI = True   # set False to skip

def _std(s):
    return (s - s.mean()) / s.std(ddof=1)

if COMPUTE_NCP_MOI:
    try:
        import xarray as xr
        base = "https://psl.noaa.gov/thredds/dodsC/Datasets/ncep.reanalysis.derived"
        # --- NCP from 500 hPa geopotential height
        hgt = xr.open_dataset(f"{base}/pressure/hgt.mon.mean.nc")["hgt"].sel(level=500)
        a = hgt.sel(lon=0,  lat=55, method="nearest").to_series()
        b = hgt.sel(lon=60, lat=55, method="nearest").to_series()
        ncp = (_std(a) - _std(b)).rename("NCP").reset_index()
        # --- MOI from sea-level pressure
        slp = xr.open_dataset(f"{base}/surface/slp.mon.mean.nc")["slp"]
        c = slp.sel(lon=3.0,  lat=36.5, method="nearest").to_series()
        d = slp.sel(lon=31.5, lat=30.0, method="nearest").to_series()
        moi = (_std(c) - _std(d)).rename("MOI").reset_index()

        extra = ncp.merge(moi, on="time")
        extra["year"]  = pd.to_datetime(extra["time"]).dt.year
        extra["month"] = pd.to_datetime(extra["time"]).dt.month
        extra = extra[["year", "month", "NCP", "MOI"]]
        monthly_idx = monthly_idx.merge(extra, on=["year", "month"], how="left")
        extra.to_csv(os.path.join(IDXDIR, "ncp_moi_ncep.csv"), index=False)
        print("NCP and MOI computed and cached")
    except Exception as e:
        print(f"skipped NCP/MOI ({type(e).__name__}: {e})")
        cache = os.path.join(IDXDIR, "ncp_moi_ncep.csv")
        if os.path.exists(cache):
            monthly_idx = monthly_idx.merge(pd.read_csv(cache), on=["year", "month"], how="left")
            print("  -> loaded from cache instead")

NCP and MOI computed and cached


## 4. Seasonal aggregation, including lags

Three aggregation windows are produced for every index:

- `_ONDJFM` - the full snow season, matched 1:1 to the snow metrics
- `_DJF`    - mid-winter core, when the accumulation signal is cleanest
- `_SON`    - the **preceding autumn**, i.e. a one-season lead. A significant `_SON`
  correlation is much harder to dismiss as coincidence than a concurrent one, because
  the predictor precedes the response.

In [5]:
IDX_COLS = [c for c in monthly_idx.columns if c not in ("year", "month")]

# season label = ending year, as in notebook 01
d = monthly_idx.copy()
d["season"] = np.where(d["month"] >= 10, d["year"] + 1, d["year"])

def window(df, months, suffix):
    sub = df[df["month"].isin(months)]
    if set(months) <= {9, 10, 11}:            # SON belongs to the season that follows
        sub = sub.assign(season=sub["year"] + 1)
    out = sub.groupby("season")[IDX_COLS].mean()
    out.columns = [f"{c}_{suffix}" for c in out.columns]
    return out

seasonal_idx = (window(d, [10, 11, 12, 1, 2, 3], "ONDJFM")
                .join(window(d, [12, 1, 2],      "DJF"),  how="outer")
                .join(window(d, [9, 10, 11],     "SON"),  how="outer")
                .reset_index())

seasonal_idx = seasonal_idx[(seasonal_idx["season"] >= 2001) &
                            (seasonal_idx["season"] <= 2024)].reset_index(drop=True)

monthly_idx.to_csv(os.path.join(OUTDIR, "indices_monthly.csv"), index=False)
seasonal_idx.to_csv(os.path.join(OUTDIR, "indices_seasonal.csv"), index=False)
print("saved indices_monthly.csv and indices_seasonal.csv")
print(seasonal_idx.shape)
seasonal_idx.head(3).round(2)

saved indices_monthly.csv and indices_seasonal.csv
(24, 49)


,season,NAO_ONDJFM,EA_ONDJFM,WP_ONDJFM,EPNP_ONDJFM,PNA_ONDJFM,EAWR_ONDJFM,SCAND_ONDJFM,TNH_ONDJFM,POL_ONDJFM,...,SCAND_SON,TNH_SON,POL_SON,PT_SON,AO_SON,NINO34_SON,DMI_SON,AMO_SON,NCP_SON,MOI_SON
0,2001,-0.35,0.98,0.19,-0.46,0.54,-0.70,0.80,0.27,-0.63,...,1.47,NaN,0.01,-0.81,-0.29,-0.50,-0.18,-0.02,-0.85,-0.54
1,2002,0.17,0.56,0.24,0.12,-0.19,0.14,-0.46,-0.28,0.18,...,-0.10,NaN,0.12,1.28,0.27,-0.01,-0.31,0.21,0.59,-0.06
2,2003,-0.56,0.66,-0.68,1.48,0.59,0.76,0.42,0.32,0.36,...,0.58,NaN,0.28,-0.56,-0.99,0.98,0.26,0.05,0.01,-0.49


In [6]:
# how complete is each seasonal predictor over 2001-2024?
cov = seasonal_idx.drop(columns=["season"]).notna().mean().sort_values()
print("Predictors with gaps (CPC patterns are only archived when they are a leading mode):")
print((cov[cov < 1.0] * 100).round(0).astype(int).to_string())
print(f"\n{(cov == 1.0).sum()} of {len(cov)} predictors are complete.")

Predictors with gaps (CPC patterns are only archived when they are a leading mode):
PT_ONDJFM      0
PT_DJF         0
TNH_SON        0
AMO_DJF       96
AMO_SON       96
AMO_ONDJFM    96

42 of 48 predictors are complete.


---
**Note for the Methods section.** The CPC patterns are only archived for months in which they
appear among the ten leading rotated EOF modes, so EA/WR, SCAND and POL have occasional gaps.
Notebook 04 drops any predictor with fewer than 20 of 24 seasons rather than interpolating.

**Next:** `04_attribution_analysis.ipynb`.